# Notebook 01 — Generador de Formularios D14 Simulados
Genera imágenes de formularios D14 con Pillow: versión limpia y versión con tachones (anomalías).

In [ ]:
import os
from PIL import Image, ImageDraw, ImageFont
import random

os.makedirs('../data', exist_ok=True)

CANDIDATOS = ['Candidato A', 'Candidato B', 'Candidato C', 'Candidato D']
MESAS = ['Mesa 001', 'Mesa 002', 'Mesa 003', 'Mesa 004', 'Mesa 005']

W, H = 700, 500
MARGEN = 40
COL_NOMBRE = 200
COL_VOTOS = 120
FILA_H = 50
HEADER_H = 80

In [ ]:
def generar_votos(anomalia=False):
    """Genera dict {candidato: votos}. Con anomalia=True inserta valor inflado."""
    votos = {c: random.randint(10, 150) for c in CANDIDATOS}
    if anomalia:
        candidato_alterado = random.choice(CANDIDATOS)
        votos[candidato_alterado] = random.randint(300, 500)  # valor sospechoso
    return votos

def calcular_total(votos):
    return sum(votos.values())

In [ ]:
def dibujar_formulario(draw, votos, mesa, con_tachon=False):
    """Dibuja tabla D14 en el objeto draw."""
    # Encabezado
    draw.rectangle([MARGEN, MARGEN, W - MARGEN, MARGEN + HEADER_H], outline='black', width=2)
    draw.text((W // 2, MARGEN + 15), 'ACTA D14 — RESULTADOS ELECTORALES',
              fill='black', anchor='mt')
    draw.text((W // 2, MARGEN + 40), f'Formulario: {mesa}  |  Fecha: 2024-11-15',
              fill='black', anchor='mt')

    # Cabecera de tabla
    y = MARGEN + HEADER_H + 10
    draw.rectangle([MARGEN, y, MARGEN + COL_NOMBRE, y + FILA_H], outline='black', width=1)
    draw.text((MARGEN + 5, y + 15), 'Candidato', fill='black')
    draw.rectangle([MARGEN + COL_NOMBRE, y, MARGEN + COL_NOMBRE + COL_VOTOS, y + FILA_H],
                   outline='black', width=1)
    draw.text((MARGEN + COL_NOMBRE + 20, y + 15), 'Votos', fill='black')

    # Filas de candidatos
    tachon_fila = random.randint(0, len(CANDIDATOS) - 1) if con_tachon else -1
    for i, (candidato, votos_val) in enumerate(votos.items()):
        y += FILA_H
        # Columna nombre
        draw.rectangle([MARGEN, y, MARGEN + COL_NOMBRE, y + FILA_H], outline='black', width=1)
        draw.text((MARGEN + 5, y + 15), candidato, fill='black')
        # Columna votos
        draw.rectangle([MARGEN + COL_NOMBRE, y, MARGEN + COL_NOMBRE + COL_VOTOS, y + FILA_H],
                       outline='black', width=1)
        draw.text((MARGEN + COL_NOMBRE + 20, y + 15), str(votos_val), fill='black')
        # Tachón sobre celda de votos
        if con_tachon and i == tachon_fila:
            x0 = MARGEN + COL_NOMBRE + 5
            x1 = MARGEN + COL_NOMBRE + COL_VOTOS - 5
            for offset in range(-4, 5, 2):
                draw.line([x0, y + 25 + offset, x1, y + 25 + offset], fill='black', width=2)
            draw.line([x0, y + 5, x1, y + FILA_H - 5], fill='black', width=2)
            draw.line([x0, y + FILA_H - 5, x1, y + 5], fill='black', width=2)

    # Fila total
    y += FILA_H
    total = calcular_total(votos)
    draw.rectangle([MARGEN, y, MARGEN + COL_NOMBRE, y + FILA_H], outline='black', width=2)
    draw.text((MARGEN + 5, y + 15), 'TOTAL', fill='black')
    draw.rectangle([MARGEN + COL_NOMBRE, y, MARGEN + COL_NOMBRE + COL_VOTOS, y + FILA_H],
                   outline='black', width=2)
    draw.text((MARGEN + COL_NOMBRE + 20, y + 15), str(total), fill='black')

In [ ]:
generados = []

for mesa in MESAS:
    for es_anomalia in [False, True]:
        votos = generar_votos(anomalia=es_anomalia)
        img = Image.new('RGB', (W, H), color='white')
        draw = ImageDraw.Draw(img)
        dibujar_formulario(draw, votos, mesa, con_tachon=es_anomalia)

        sufijo = 'anomalo' if es_anomalia else 'limpio'
        nombre = f'../data/d14_{mesa.replace(" ", "_")}_{sufijo}.png'
        img.save(nombre)
        generados.append({'archivo': nombre, 'mesa': mesa,
                          'anomalia': es_anomalia, 'votos': votos,
                          'total': calcular_total(votos)})
        print(f'Guardado: {nombre}')

print(f'\nTotal formularios generados: {len(generados)}')

In [ ]:
# Visualizar un ejemplo de cada tipo
from IPython.display import display

img_limpio = Image.open('../data/d14_Mesa_001_limpio.png')
img_anomalo = Image.open('../data/d14_Mesa_001_anomalo.png')

print('Formulario LIMPIO:')
display(img_limpio)
print('Formulario CON TACHÓN:')
display(img_anomalo)